# LLM Fundamentals

> 📘 **Python Mastery** · Module 17 — LLM Engineering · Lesson 1/7

Large language models look like magic but rest on one simple trick — predicting
the next token — scaled up enormously. Before you touch APIs, RAG, or agents,
you need this mental model: what tokens are, how these systems are trained,
how sampling shapes output, and how to pick the right model for the job.

## 🎯 Learning Objectives

- Explain what an LLM actually computes at inference time (next-token prediction in a loop)
- Convert character counts into estimated tokens and dollars using a real price table
- Trace the three training stages — pretraining → supervised fine-tuning → RLHF — and what each one changes
- Implement softmax-with-temperature and top-p filtering in NumPy and predict their effect on a distribution
- Choose between Claude Haiku / Sonnet / Opus tiers using a task × volume × budget grid
- Argue the open-weight vs closed-weight tradeoff for a concrete product scenario

## 1. What Is an LLM, Really?

A Large Language Model is a **next-token predictor**. Give it a sequence of text
and it returns a probability for every token in its vocabulary (~100k+
candidates). Generating an answer is nothing more than feeding that prediction
back in and looping. Essays, code, and jokes all emerge from repeating this one
step extremely well.

Analogy: the world's smartest autocorrect. It does not *look up* answers —
it *continues* text so plausibly that the continuation is usually useful.

**Syntax:** *(pseudocode — the entire generation process)*

```python
context = ["The", "capital", "of", "France", "is"]
while not done:
    probs   = model(context)          # probability for EVERY vocabulary token
    next_id = sample_or_argmax(probs) # greedy pick, or weighted random sample
    context.append(next_id)           # feed prediction back in -> loop
```

In [ ]:
import random

random.seed(42)                        # seeded => same picks on every run

# A toy "language model": hand-written next-word distributions.
TOY_LM = {
    "the cat sat on the":     {"mat": 0.55, "roof": 0.30, "keyboard": 0.15},
    "good morning":           {"world": 0.60, "team": 0.25, "sunshine": 0.15},
    "i love programming in ": {"python": 0.70, "rust": 0.20, "cobol": 0.10},
}

for prompt, dist in TOY_LM.items():
    word = random.choices(list(dist), weights=list(dist.values()))[0]  # one weighted draw
    print(f"{prompt!r} -> {word!r}   (candidates: {dist})")

# A real LLM repeats EXACTLY this step once per generated token --
# with ~100,000 candidate tokens instead of three.

## 2. Tokens: How Text Becomes Numbers — and Money

Models never see characters or words; they see **tokens**, subword chunks
produced by Byte-Pair Encoding. You met BPE in Module 16 (NLP) — here we care
about the practical consequences:

- **Rule of thumb:** 1 token ≈ 4 characters ≈ ¾ of an English word.
- Pricing, context limits, and rate limits are all quoted **per token**.
- Machine-format text (JSON, code, long numbers) packs more tokens per character than prose.

**Syntax:**

```python
est_tokens = len(text) / 4                                     # cheap heuristic (English)
cost = input_tokens/1e6*price_in + output_tokens/1e6*price_out  # the whole billing model
```

> 🔍 **Under the Hood:** BPE starts from raw bytes and greedily merges the most
> frequent adjacent pair, again and again, until a target vocabulary size is
> reached. Frequent words fuse into single tokens (`" the"`, `" def"`), rare
> words shatter into pieces, whitespace usually glues onto the token *after*
> it, and each digit of a long number tends to split off individually — one
> reason LLMs wobble at arithmetic on big numbers: they never see the number as
> one unit.

In [ ]:
import math

# US$ per 1M tokens (input, output) -- Anthropic price table.
PRICES_PER_1M = {
    "claude-opus-5":    (5.00, 25.00),
    "claude-sonnet-5":  (2.00, 10.00),
    "claude-haiku-4-5": (1.00,  5.00),
}

def est_tokens(text):
    """Heuristic token estimate: ~4 characters per token of English prose."""
    return math.ceil(len(text) / 4)

prompt_text = ("Summarise the following customer review in two sentences, then "
               "rate its sentiment from 1 to 5: " + "Great laptop, battery lasts all day. " * 60)
out_tokens = 80                                        # assume a short summary reply

print(f"Prompt characters : {len(prompt_text)}")
print(f"Estimated tokens  : ~{est_tokens(prompt_text)}")
print("-" * 58)
for model, (p_in, p_out) in PRICES_PER_1M.items():
    t_in = est_tokens(prompt_text)
    cost = t_in / 1e6 * p_in + out_tokens / 1e6 * p_out
    print(f"{model:<18} ~{t_in:>4} in-tok + {out_tokens} out-tok -> ${cost:.6f}/request")

In [ ]:
# The /4 heuristic is calibrated on PROSE. Other text types differ a lot.
samples = {
    "English prose": "The quick brown fox jumps over the lazy dog near the river bank.",
    "JSON payload":  '{"user_id": 8842, "active": true, "roles": ["admin", "editor"]}',
    "Python code":   "def f(x):\n    return sum(v * v for v in x if v % 2 == 0)",
    "Long number":   "4815162342951387",
}
print(f"{'text type':<14} {'chars':>5} {'words':>5} {'est tokens':>11}")
for label, text in samples.items():
    est = max(1, round(len(text) / 4))
    print(f"{label:<14} {len(text):>5} {len(text.split()):>5} {est:>11}")

# Dense machine text carries MORE tokens per character, so the /4 rule
# UNDERSTATES its true count. Treat the heuristic as a floor -- measure with
# client.messages.count_tokens (lesson 3) when money is involved.

## 3. How an LLM Is Made: Pretraining → SFT → RLHF

Every commercial model passed through three stages. Each stage changes the
model's *behaviour* — what it is willing to do and how it responds — more than
it changes raw knowledge.

| Stage | Data | The model learns | Result |
|---|---|---|---|
| **Pretraining** | Trillions of tokens: web pages, books, code | Predict the next token, everywhere | A raw autocomplete engine — vast knowledge, no manners |
| **Supervised Fine-Tuning (SFT)** | Curated `(instruction → ideal answer)` demonstrations | Follow instructions, converse, respect formats | An assistant that actually answers you |
| **RLHF / preference tuning** | Human-ranked answer pairs ("A is better than B") | Prefer helpful, honest, harmless continuations | Polished tone, refusals, calibrated style |

**Syntax:** *(the pipeline as code)*

```python
base      = pretrain(corpus="trillion_tokens_web")            # learns LANGUAGE + WORLD
assistant = sft(base, examples=[("question", "ideal_answer"), ...])  # learns to OBEY
aligned   = rlhf(assistant, pairs=[("better_answer", "worse_answer")])  # learns TASTE
```

In [ ]:
# SIMULATED: what each training stage contributes (canned outputs, no model).
# Notice: knowledge was already there after stage 1 -- later stages shape BEHAVIOUR.
stages = {
    "After pretraining only":
        "reset my password reset password forgot my password login page ...",
    "After SFT":
        "Sure! Go to Settings > Security > 'Change password', enter your current "
        "password, then pick a new one.",
    "After RLHF":
        "Happy to help! Head to Settings > Security > 'Change password'. Tip: use a "
        "long passphrase. (And I can't help access someone else's account.)",
}
for stage, answer in stages.items():
    print(f"{stage}:\n   {answer}\n")

## 4. One-Screen Transformer Recap

Module 16 (NLP) covered transformers in depth; here is the 60-second refresher
that matters for LLM engineering:

1. Tokens map to embedding vectors, plus positional information (order matters).
2. Each of N stacked blocks applies **self-attention** — every token gathers
   information from earlier tokens ("who is relevant to whom?") — then an **MLP**
   where each token thinks privately about what it gathered.
3. Attention is **causal**: token *t* may only look left. That is precisely why
   generation can proceed one token at a time.
4. A final projection maps the last vector to a score for every vocabulary token.

**Syntax:**

```python
import numpy as np
W = softmax(Q @ K.T / np.sqrt(d_head))   # the who-looks-at-whom matrix
context = W @ V                          # each token pulls in weighted context
```

**Example:** *(a miniature causal attention pass you can run)*

In [ ]:
import numpy as np

rng = np.random.default_rng(42)                # seeded => reproducible

seq_len, d_model, d_head = 4, 16, 8            # tiny dims (real models: 100k+ tokens, d~16k)
tokens = ["The", "cat", "sat", "down"]

X = rng.normal(size=(seq_len, d_model))        # stand-in for token embeddings
Wq, Wk, Wv = (rng.normal(size=(d_model, d_head)) for _ in range(3))

Q, K, V = X @ Wq, X @ Wk, X @ Wv
scores = Q @ K.T / np.sqrt(d_head)             # raw compatibility scores
future = np.triu(np.ones((seq_len, seq_len)), k=1).astype(bool)
scores[future] = -1e9                          # mask FUTURE positions (causality)

w = np.exp(scores - scores.max(axis=-1, keepdims=True))   # stable softmax, row-wise
w /= w.sum(axis=-1, keepdims=True)

print("Attention weights (row = token attending, cols = The/cat/sat/down):")
for t, row in zip(tokens, w.round(2)):
    print(f"  {t:<5} -> {row}")
print("\nContext vectors:", (w @ V).shape, "-- one blended vector per token")

## 5. Capabilities vs Limits

LLMs are astonishingly broad — and confidently wrong in specific, *predictable*
ways. Engineering with LLMs means designing around the limits, not hoping away.

**Capabilities:** drafting and rewriting · summarisation · translation · code
generation and review · extracting structured data · classification · reasoning
over supplied text · orchestrating tools (lesson 7).

| Limit | Mechanism — why it happens | Mitigation |
|---|---|---|
| **Hallucination** | Training rewards *plausible* next tokens, not *true* ones; no fact-checker exists inside the forward pass | Ground in retrieved documents (RAG, lesson 5); demand citations; validate outputs |
| **Knowledge cutoff** | Weights are frozen at training time | Fetch fresh data at query time; never trust the model for recent facts |
| **Context window vs cost** | You pay for every token on every call; very long contexts also dilute attention | Lean prompts, cached prefixes (lesson 3), retrieve instead of dump |
| **Reasoning ≠ knowledge** | Strong at transforming stated premises; cannot conjure unstated private facts | Reasoning → capable model / thinking mode; knowledge → retrieval & tools |

Hallucination is worth a closer look because it is *mechanistic*. When the model
does not know, its distribution goes flat over many equally-plausible
continuations — and sampling from a flat distribution still yields perfectly
fluent text.

**Syntax:**

```python
entropy = -(probs * np.log(probs)).sum()   # high entropy = the model is guessing
```

**Example:** *(measure the difference between knowing and guessing)*

In [ ]:
import numpy as np

def entropy(probs):
    """Shannon entropy: 0 = certain, higher = more uncertain."""
    p = np.asarray(list(probs.values()), dtype=float)
    return float(-(p * np.log(p)).sum())

knows   = {"Shakespeare": 0.94, "Marlowe": 0.04, "Dickens": 0.02}          # "Hamlet was written by ..."
guesses = {"555-0134": 0.25, "555-0198": 0.25, "555-0117": 0.25, "555-0162": 0.25}

for label, dist in [("confident (knows)", knows), ("flat (guessing)", guesses)]:
    top = max(dist, key=dist.get)
    print(f"{label:<19} top pick = {top!r:<14} entropy = {entropy(dist):.3f}")

# Both cases produce equally fluent sentences. The confidence gap is invisible
# to a reader unless you MEASURE it -- which is why production systems surface
# citations, logprobs, or verification instead of trusting the tone.

## 6. Sampling: Greedy, Temperature, and top_p

The network ends with **logits** (one raw score per token). Decoding turns
scores into a choice:

- **Greedy decoding** — always take the argmax. Deterministic; ideal for tests, extraction, structured output.
- **Temperature** — divide logits by `T` before the softmax. `T < 1` sharpens the
  distribution (safe, repetitive), `T = 1` is the model's native distribution,
  `T > 1` flattens it (creative, riskier).
- **top_p (nucleus sampling)** — keep only the smallest set of tokens whose
  cumulative probability reaches `p`, renormalise, sample from that set. Prunes
  the long tail of nonsense regardless of temperature.

**Syntax:**

```python
def softmax_with_temperature(logits, T=1.0):
    z = np.asarray(logits, dtype=float) / T
    z -= z.max()                     # numeric-stability trick (see below)
    e = np.exp(z)
    return e / e.sum()

def top_p_keep(probs, p=0.9):        # boolean mask: True = token survives
    ...
```

> 🔍 **Under the Hood:** `z -= z.max()` before `np.exp` is not decoration.
> `exp()` overflows around argument ≈ 700, producing `inf` and a `nan` softmax;
> subtracting the maximum makes every exponent ≤ 0, so the top token is exactly
> `exp(0) = 1` and the result is mathematically unchanged (constant shift cancels
> in the ratio). Every serious inference stack does this — and the same trick
> reappears in cross-entropy loss and in the attention scores above.

In [ ]:
import numpy as np

def softmax_with_temperature(logits, T=1.0):
    z = np.asarray(logits, dtype=float) / T
    z -= z.max()                              # stability: largest exponent is now 0
    e = np.exp(z)
    return e / e.sum()

words  = ["mat", "roof", "sofa", "quantum"]   # next word after "The cat sat on the"
logits = np.array([3.0, 2.0, 1.0, 0.0])

for T in (0.1, 1.0, 2.0):
    probs = softmax_with_temperature(logits, T)
    pretty = ", ".join(f"{w}:{p:.3f}" for w, p in zip(words, probs))
    print(f"T={T:>4}: {pretty}")

# T=0.1 -> nearly all mass on "mat" (greedy in disguise).
# T=1.0 -> the model's honest opinion.
# T=2.0 -> even "quantum" gets real odds: creativity, plus risk.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)               # seeded => reproducible draws

words  = np.array(["mat", "roof", "sofa", "quantum"])
logits = np.array([3.0, 2.0, 1.0, 0.0])

def softmax_with_temperature(logits, T=1.0):
    z = np.asarray(logits, dtype=float) / T
    z -= z.max(); e = np.exp(z)
    return e / e.sum()

def top_p_keep(probs, p=0.9):
    """Mask keeping the smallest highest-probability set summing to >= p."""
    order = np.argsort(probs)[::-1]
    keep, cum = [], 0.0
    for i in order:
        keep.append(int(i)); cum += probs[i]
        if cum >= p:
            break
    mask = np.zeros(len(probs), dtype=bool); mask[keep] = True
    return mask

probs    = softmax_with_temperature(logits, T=1.0)
mask     = top_p_keep(probs, p=0.90)
filtered = np.where(mask, probs, 0.0)
filtered /= filtered.sum()                    # renormalise survivors

print("survivors after top_p=0.9 :", words[mask].tolist())
print("five sampled continuations:")
for _ in range(5):
    print("   ", rng.choice(words, p=filtered))

## 7. Choosing a Model: Task × Volume × Budget

Anthropic ships three tiers sharing one identical API — switching is a
one-string change — so match tier to workload, not to hype.

| Model | Input $/1M tok | Output $/1M tok | Context | Sweet spot |
|---|---|---|---|---|
| `claude-haiku-4-5` | $1 | $5 | 200K | High-volume simple work: classification, extraction, routing, autocompletion |
| `claude-sonnet-5`  | $2 | $10 | 200K | The production default — balanced quality/speed/cost for most features |
| `claude-opus-5`    | $5 | $25 | 1M | The hardest tasks: deep reasoning, huge documents, agentic planning |

*(Prices and context sizes move with releases — re-check the docs when budgeting.)*

Decision grid:

| Volume ↓ \ Complexity → | Mechanical / simple | Medium | Hard / agentic |
|---|---|---|---|
| **Millions of calls/day** | Haiku | Haiku, Sonnet spot-checks | Sonnet default + Opus escalation |
| **Thousands of calls/day** | Haiku | Sonnet | Opus |
| **Prototype / low volume** | Sonnet | Sonnet | Opus |

Popular pattern: **model cascades** — try the cheap tier first and escalate only
when quality checks fail.

**Syntax:**

```python
monthly_cost = n_requests * (in_tokens/1e6 * price_in + out_tokens/1e6 * price_out)
```

**Example:** *(run the numbers for the same feature on all three tiers)*

In [ ]:
REQUESTS_PER_DAY = 50_000
IN_TOKENS, OUT_TOKENS = 400, 150             # typical classify-and-summarise traffic
DAYS = 30

PRICES = {"claude-haiku-4-5": (1, 5), "claude-sonnet-5": (2, 10), "claude-opus-5": (5, 25)}

requests = REQUESTS_PER_DAY * DAYS
print(f"{requests:,} requests/month, {IN_TOKENS} in + {OUT_TOKENS} out tokens each\n")
for model, (p_in, p_out) in PRICES.items():
    monthly = requests * (IN_TOKENS / 1e6 * p_in + OUT_TOKENS / 1e6 * p_out)
    print(f"{model:<18} ${monthly:>9,.0f} / month")

# A 5x price spread between Haiku and Opus on identical traffic.
# Earn the expensive tier with evaluation results -- never assume it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

daily = np.linspace(1_000, 100_000, 200)     # requests per day
MODELS = {"claude-haiku-4-5": (1, 5), "claude-sonnet-5": (2, 10), "claude-opus-5": (5, 25)}
IN_TOK, OUT_TOK, DAYS = 400, 150, 30

plt.figure(figsize=(7, 4.2))
for name, (p_in, p_out) in MODELS.items():
    per_request = IN_TOK / 1e6 * p_in + OUT_TOK / 1e6 * p_out
    monthly_k = daily * DAYS * per_request / 1000
    plt.plot(daily, monthly_k, label=name)
plt.xlabel("Requests per day")
plt.ylabel("Monthly cost ($ thousands)")
plt.title("Cost of one feature across model tiers (400 in / 150 out tokens)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Open vs Closed Weights

**Closed models** (the Claude family) are served by the vendor through an API:
frontier quality, zero infrastructure, instant upgrades — in exchange for data
leaving your network and renting rather than owning behaviour.

**Open-weight models** (Llama, Mistral, Qwen, …) publish downloadable
parameters you can host anywhere: full privacy, arbitrary fine-tuning,
predictable cost at massive volume — paid for with GPU operations and usually
somewhat lower peak quality.

| Dimension | Closed API | Open weights |
|---|---|---|
| Quality ceiling | Frontier | Good, typically trails the frontier |
| Data privacy | Governed by vendor terms | Nothing leaves your VPC |
| Ops burden | None | GPUs, serving, scaling — yours |
| Customisation | Prompting, caching, selective fine-tuning | Anything, down to the raw weights |
| Cost shape | Per-token forever | Fixed infrastructure, near-zero marginal |
| Upgrades | Automatic | You rebuild and redeploy each release |

**Syntax:** *(encode the tradeoff as a decision helper)*

```python
def recommend_route(data_sensitive, has_gpu_ops, daily_requests): ...
```

**Example:** *(run the helper on three realistic products)*

In [ ]:
def recommend_route(data_sensitive, has_gpu_ops, daily_requests):
    if data_sensitive and not has_gpu_ops:
        return "closed API + enterprise data agreement (revisit on-prem later)"
    if data_sensitive and has_gpu_ops:
        return "self-hosted open weights, fine-tuned, inside your VPC"
    if daily_requests > 2_000_000:
        return "hybrid: open weights for bulk traffic, closed API for hard cases"
    return "closed API -- iterate fast, re-evaluate at scale"

scenarios = [
    ("hospital notes summariser", True,  False,     40_000),
    ("public docs chatbot",       False, False,     12_000),
    ("ad-click classifier",       True,  True,   5_000_000),
    ("startup support triage",    False, False,    900_000),
]
for name, sensitive, gpu, volume in scenarios:
    verdict = recommend_route(sensitive, gpu, volume)
    print(f"{name:<28} -> {verdict}")

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Estimating cost from characters alone | Code, JSON, and numbers tokenize much denser than 4 chars/token — invoices surprise you | Use `client.messages.count_tokens` (lesson 3) for anything billable; keep ÷4 only for napkin math |
| Budgeting input tokens only | Output tokens cost 5× input at every tier — verbose replies dominate the bill | Set sane `max_tokens`, monitor `usage.output_tokens`, compress outputs |
| Running evaluations with `temperature=1.0` | Tests pass and fail randomly; CI flakes; regression signals vanish | Greedy / low temperature for evals and extraction; save warmth for ideation |
| Asking the model about post-cutoff events | Confident, detailed, wrong answers | Retrieve live data (lesson 5); state today's date in the prompt |
| Defaulting to the biggest model everywhere | 5–25× cost for quality differences you probably cannot measure | Benchmark cheaper tiers against your golden set; escalate deliberately |

## 💡 Best Practices & Pro Tips

- Think in **tokens** everywhere — prompts, budgets, rate limits, invoices. Every number in LLM engineering is denominated in tokens.
- Record the model ID (and ideally version) alongside every output; silent provider upgrades otherwise make old evaluations incomparable.
- Build a small "cost per feature" table early — it settles caching, model-tier, and prompt-length debates faster than opinions.
- Treat hallucination as a **systems** problem: ground with retrieval, constrain with schemas, verify with checks. Prompting alone only reduces it.
- Adopt cascades: cheapest tier first, escalate on measured failure. The reverse habit quietly doubles cloud spend.
- **AI-engineering relevance:** the softmax/temperature and attention math you just ran is the exact machinery reused later — decoding params in API calls (lesson 3), cosine similarity in vector stores (lesson 4), and logit-level behaviour during fine-tuning (lesson 6).

## 📌 Summary

| Concept | Key idea | Example |
|---|---|---|
| Next-token prediction | LLM = probability over ~100k tokens, fed back in a loop | `sample_or_argmax(model(context))` |
| Tokens | Subword units, ≈4 chars each; the currency of pricing and limits | `len(text) // 4` heuristic |
| Training pipeline | Pretraining → SFT → RLHF: language → obedience → taste | Raw autocomplete becomes an assistant |
| Attention | Causal, weighted mixing of earlier token vectors | `softmax(QKᵀ/√d) @ V` |
| Temperature / top_p | Sharpen or flatten the distribution before sampling | `T=0.1` near-greedy, `T=2.0` adventurous |
| Model tiers | Haiku $1/$5 · Sonnet $2/$10 · Opus $5/$25 per 1M in/out tokens | Route by task × volume × budget |
| Open vs closed | Privacy + control vs quality + zero ops | Hybrid cascade at scale |

Key takeaways:
- Everything an LLM produces is token-level probability computation — fluency is guaranteed, truth is not.
- Costs are token-shaped: estimate with heuristics, verify with `count_tokens`, cap with `max_tokens`.
- Decoding settings are product decisions: deterministic for pipelines, warmer for people.
- Model choice is an economics problem — measure quality first, then pay only for capability you can detect.

## 🔗 Next Lesson

Continue with [`../02_Prompt_Engineering/notes.ipynb`](../02_Prompt_Engineering/notes.ipynb) —
turning these fundamentals into reliable, injection-resistant, evaluated prompts.